In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Forecast')

import os
import numpy as np
import pandas as pd
import random
import json

import plot_dashes

import processing as prc

import models

In [5]:
# # Opção usando NumPy - dirichlet

# rng = np.random.default_rng()
# rng.dirichlet([1, 1, 1], size=20)

# Inicialização - Opção usando código na mão
def start_population(max_size):
    '''
    20 Individuos com espaco de busca de 3 dimensões (3 modelos), 
    de pesos continuos, com valores entre 0 e 1, e a soma dos 
    pesos deve ser igual a 1.
    
    '''
    population = []
    epsilon = 1e-6

    for _ in range(max_size):

        ind = []

        first_element = random.uniform(epsilon, 0.98)
        second_element = random.uniform(epsilon, 1 - first_element - epsilon)
        third_element = 1 - (first_element + second_element)

        ind = [first_element, second_element, third_element]
        random.shuffle(ind)

        population.append(ind)

    return population

# 2. Avaliação

# Aplica-se uma função de fitness no conjunto.
# Essa função define o quão boa é cada solução (quanto maior ou menor, melhor — depende do problema).
#     * Nesse processo, os individuos inicializados serão executados em cada modelo e avaliados de acordo com o resultado do fitness utilizado.

start_population(20)


[[0.1065765800354961, 0.884196356312828, 0.009227063651675915],
 [0.07693075270832014, 0.8124085317954954, 0.11066071549618439],
 [0.10069063041181298, 0.7946550483880417, 0.10465432120014537],
 [0.5213790060164403, 0.38899681567043887, 0.08962417831312088],
 [0.9522931835724492, 0.007202242995248807, 0.04050457343230196],
 [0.8850138205602276, 0.004691913112480739, 0.11029426632729171],
 [0.5441319982843059, 0.07508974177832695, 0.3807782599373672],
 [0.3206683739419469, 0.6715643654705583, 0.007767260587494812],
 [0.2375463647263193, 0.03598645033973364, 0.726467184933947],
 [0.0730802871305209, 0.7622435425682212, 0.1646761703012579],
 [0.10633501067297531, 0.03203243759588237, 0.8616325517311423],
 [0.23363859391897615, 0.26422241437868055, 0.5021389917023433],
 [0.9179151939413513, 0.06296182997444244, 0.01912297608420621],
 [0.3267544134074799, 0.28320038310736184, 0.39004520348515825],
 [0.213013798114305, 0.1687710820877591, 0.618215119797936],
 [0.12347640386040781, 0.05796384

In [ ]:
metrics_lst = []

def get_file():

    file_path = '/content/drive/MyDrive/Colab Notebooks/Forecast/Datasets'
    arquivos = []

    for f in os.listdir(file_path):
        full_path = os.path.join(file_path, f)

        yield full_path
     

for file_content in get_file():

    X = pd.read_csv(file_content, sep='\t', decimal='.')

    data = X.iloc[:, 0].values

    n_lags = 5
    X, y = prc.window_lags(data, n_lags)

    # MLP Params
    solver = 'sgd' # adam, sgd
    hidden_neurons = [10,20,30,40]
    learning_rate = [0.1, 0.01, 0.001]
    activation=['logistic','tanh','relu']
    #[[10, 0.1, 'relu'],[10, 0.1, 'logistic']... [40, 0.001, 'tanh']]
    jumps =  10

    lst_results, pred_test, pred_test_denom, X_test, y_test, best_rna = models.forecast_mlp(
        X,
        y,
        solver,
        hidden_neurons,
        learning_rate,
        activation,
        jumps
    )

    # MLP Metrics
    metrics_error = plot_dashes.get_metrics_error(y_test, pred_test_denom)
    metrics = {
            'base': file_content,
            'data': {
                'model': 'MLP',
                'metrics': metrics_error
            }
        }
    metrics_lst.append(metrics)

    # SVR
    C_values=[0.1, 1, 10, 100] # Penalidade por erro
    epsilon_values=[0.01, 0.05, 0.1]
    gamma_values=['scale', 0.01, 0.1]

    lst_results, pred_test, X_test, y_test, best_svr = models.forecast_svr(X, y, C_values, epsilon_values, gamma_values)

    # SVR Metrics
    metrics_error = plot_dashes.get_metrics_error(y_test, pred_test)
    metrics = {
            'base': file_content,
            'data': {
                'model': 'SVR',
                'metrics': metrics_error
            }
        }
    metrics_lst.append(metrics)

    # RF
    n_estimators_values = [50, 100, 200]
    max_depth_values = [None, 5, 10, 20]

    lst_results, pred_test, X_test, y_test, best_rf = models.forecast_rf(X, y, n_estimators_values, max_depth_values)

    # RF Metrics
    metrics_error = plot_dashes.get_metrics_error(y_test, pred_test)
    metrics = {
            'base': file_content,
            'data': {
                'model': 'RF',
                'metrics': metrics_error
            }
        }
    metrics_lst.append(metrics)
    

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro

In [17]:
svr

array([2713, 3800, 3091, 2985, 3790,  674,   81,   80,  108,  229,  399,
       1132, 2432, 3574, 2935, 1537,  529,  485,  662, 1000, 1590, 2657,
       3396])

In [33]:
to_json = json.dumps(metrics_lst, indent=4)
content = json.loads(to_json)